### 토큰 수 측정
- 파일 : nA1_tokenizer.ipynb
- 역할 : 토크나이저로 문장의 토큰 수를 직접 세어 과금·지연·문맥의 공통 단위를 익힙니다.
- 기능 :
    1. 토크나이저 불러와 문장 분해하기  
    2. 언어별 토큰 수 비교  
    3. 채팅 형식이 더하는 토큰  
    4. 대화 이력의 입력 토큰 누적  


# 1장 LLM 추론과 토큰의 경제학: 토크나이저로 토큰 수 측정

부록 원고의 코드와 동일합니다. 핵심 수식은 코드의 주석에 함께 적어 두었습니다.

>> [코드 A1-1] 토크나이저 불러와 문장 분해하기 (nA1_tokenizer.ipynb, STEP 1)
<hr>

In [1]:
# --------------------------------------------------------------------------------------
# 주요 기능 : 토크나이저로 문장을 토큰 ID단위로 분해하고 토큰 수와 실제 토큰 조작 확인
# 처리 흐름 : 입력과 설정을 준비한 뒤 핵심 연산을 수행하고, 출력/파일/성능 확인
# --------------------------------------------------------------------------------------

#- 패키지 설치
!pip install -q transformers

#- 모듈 로딩
from transformers import AutoTokenizer

#- LLM 모델
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

#- 모델에서 사용된 토크나이저 로딩
tok = AutoTokenizer.from_pretrained(MODEL_ID)

#- 문장을 토큰으로 분해하고 복원
text = "온디바이스 AI로 비용을 줄입니다"
ids = tok.encode(text)
print("토큰 수:", len(ids))
print("ID 목록:", ids)
print("조각들:", tok.convert_ids_to_tokens(ids))
print("복원:", tok.decode(ids))

# 확인 포인트 --------------------------------------------------------------------------
# - 조각 수와 어절 수의 불일치 확인
# - decode 결과와 원문 일치 체크

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

토큰 수: 13
ID 목록: [130000, 89235, 130271, 12802, 24897, 15235, 17380, 73986, 26699, 17877, 53989, 226, 78952]
조각들: ['ìĺ¨', 'ëĶĶ', 'ë°Ķ', 'ìĿ´', 'ìĬ¤', 'ĠAI', 'ë¡ľ', 'Ġë¹Ħ', 'ìļ©', 'ìĿĦ', 'Ġì¤', 'Ħ', 'ìŀħëĭĪëĭ¤']
복원: 온디바이스 AI로 비용을 줄입니다


>> [코드 A1-2] 언어별 토큰 수 비교 (STEP 2)
<hr>

In [3]:
#--------------------------------------------------------------------------------------
# 주요 기능 : 여러 언어의 같은 의미 문장을 토큰화해 언어에 따른 토큰 수 차이 비교
# 처리 흐름 : 입력과 설정을 준비한 뒤 핵심 연산을 수행하고, 출력/파일/성능 확인
#--------------------------------------------------------------------------------------

#- 같은 뜻의 한·영 문장 쌍
pairs = [
    ("온디바이스 AI는 기기 안에서 추론합니다.",
     "On-device AI runs inference inside the device."),
    ("메모리 대역폭이 토큰 속도의 상한을 정합니다.",
     "Memory bandwidth sets the ceiling of token speed."),
]

#- 언어별 토큰 수 비교 (빈칸 채우기)
for ko, en in pairs:
    # 빈칸: 각 문장의 토큰 수를 세는 식을 채웁니다
    n_ko, n_en = len(________), len(________)
    print(f"한국어 {n_ko}토큰 | 영어 {n_en}토큰 | 비율 {n_ko/n_en:.2f}배")

# 확인 포인트 -------------------------------------------------------------------------
# - 한국어와 영어의 토큰 수, 문장별 비율 비교
# - 같은 요금표라도 언어에 따라 실질 비용이 달라지는 이유를 한 문장으로 정리

한국어 15토큰 | 영어 9토큰 | 비율 1.67배
한국어 22토큰 | 영어 9토큰 | 비율 2.44배


>> [코드 A1-3] 채팅 형식이 더하는 토큰 (STEP 3)
<hr>

In [9]:
#--------------------------------------------------------------------------------------
# 주요 기능 : 채팅 템플릿을 적용하기 전후의 토큰 수 비교
#             시스템/역할 구분자가 추가하는 오버헤드 확인

#--------------------------------------------------------------------------------------

#- 같은 문장을 채팅 형식으로 감싸기
msgs = [{"role": "user", "content": text}]
chat_ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                   tokenize=True, return_dict=True)["input_ids"]
print(f"문장만: {len(tok.encode(text))}토큰, 채팅 형식: {len(chat_ids)}토큰")
print(tok.decode(chat_ids))

# 확인 포인트 -------------------------------------------------------------------------
# - 추가된 토큰 수와 decode 출력의 특수 토큰 확인

문장만: 13토큰, 채팅 형식: 42토큰
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
온디바이스 AI로 비용을 줄입니다<|im_end|>
<|im_start|>assistant



>> [코드 A1-4] 대화 이력의 입력 토큰 누적 (STEP 4)
<hr>

In [10]:
#--------------------------------------------------------------------------------------
# 주요 기능 : 대화 턴이 누적될수록 입력 토큰 수가 어떻게 증가하는지 계산
#--------------------------------------------------------------------------------------

#- 회차 가정: 같은 발화 반복, 답은 150 토큰
# 매 회차 사용자 발화
turn_user = "양자화가 왜 토큰 속도를 올리는지 설명해 주세요."

# 매 회차 답 길이 가정 (출력 토큰)
turn_answer_tokens = 150

#- 회차마다 전체 이력이 입력으로 들어감
msgs, total_in = [], 0
for turn in range(1, 11):
    msgs.append({"role": "user", "content": turn_user})
    #n_in = len(tok.apply_chat_template(msgs, add_generation_prompt=True))
    n_in = len(tok.apply_chat_template(msgs, add_generation_prompt=True,
                                       tokenize=True, return_dict=True)["input_ids"])
    total_in += n_in
    msgs.append({"role": "assistant", "content": "답" * turn_answer_tokens})
    if turn in (1, 5, 10):
        print(f"{turn:2d}회차 입력 {n_in:5d}토큰 | 누적 입력 {total_in:6d}토큰")

# 확인 포인트 -------------------------------------------------------------------------
# - 회차 증가에 따른 입력 토큰 증가 추이 확인
# - 10회차 누적 입력과 사용자 발화 총량의 배수 계산
# - 캐시된 입력 할인(비용 비교 참고)이 효과적인 이유 정리

 1회차 입력    53토큰 | 누적 입력     53토큰
 5회차 입력   789토큰 | 누적 입력   2105토큰
10회차 입력  1709토큰 | 누적 입력   8810토큰
